<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/QED_Feynman_Diagrams_Animations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QED: The Light & Matter Universe

This notebook contains a high-definition animation script that visualizes the fundamental interactions of **Quantum Electrodynamics (QED)** using Feynman diagram conventions. QED is the relativistic quantum field theory of electrodynamics, describing how light and matter interact.

## Visualized Phenomena

The animation cycles through six key physical processes:

1.  **Compton Scattering**: A photon hits an electron, resulting in an energy shift and change in direction.
2.  **Electron-Positron Annihilation**: An electron and its antiparticle (positron) collide to produce high-energy photons ($E=mc^2$).
3.  **Pair Production**: A high-energy photon interacts with the electric field of an atomic nucleus to create an electron-positron pair.
4.  **Bremsstrahlung (Braking Radiation)**: A charged particle (electron) is deflected by a nucleus and radiates energy as a photon.
5.  **Møller Scattering**: The interaction and repulsion between two electrons via the exchange of a virtual photon.
6.  **Vacuum Polarization**: A process where a photon temporarily fluctuates into a virtual electron-positron pair, affecting the vacuum's permittivity.

## Requirements
- `numpy`: For trajectory and wave calculations.
- `matplotlib`: For the rendering engine and animation framework.
- `ffmpeg`: Required for saving the high-quality MP4 output.

## Usage
Simply run the code cell below. It will generate an MP4 file named `qed_light_matter.mp4`, display it directly in the notebook, and automatically prompt a download.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import HTML
from base64 import b64encode
from google.colab import files

"""
QED: The Light & Matter Universe
--------------------------------
This script generates a high-definition animation illustrating six fundamental
Quantum Electrodynamics (QED) interactions using Feynman diagram conventions.
Scenes: Compton Scattering, Annihilation, Pair Production, Bremsstrahlung,
Møller Scattering, and Vacuum Polarization.
"""

# --- Constants ---
FPS = 30
DURATION_PER_SCENE = 9
NUM_SCENES = 6
TOTAL_FRAMES = FPS * DURATION_PER_SCENE * NUM_SCENES

COLORS = {
    'bg': '#0a0a1a', 'electron': '#4fc3f7', 'positron': '#ef5350',
    'photon': '#ffd54f', 'vertex': '#ffffff', 'force': '#81c784',
    'nucleus': '#808080', 'text': '#ffffff'
}

fig, ax = plt.subplots(figsize=(12.8, 7.2), dpi=100)
fig.patch.set_facecolor(COLORS['bg'])

def get_sine_wave(x0, y0, x1, y1, amplitude=0.2, freq=12, p=1.0, reverse=False):
    dist = np.hypot(x1 - x0, y1 - y0)
    angle = np.arctan2(y1 - y0, x1 - x0)
    t_vals = np.linspace(0, dist, 201)
    if reverse:
        mask_limit = dist * (1 - p)
        t = t_vals[t_vals >= mask_limit]
    else:
        t = t_vals[t_vals <= dist * p]

    wave_x = t
    wave_y = amplitude * np.sin(freq * t)
    rot_x = wave_x * np.cos(angle) - wave_y * np.sin(angle) + x0
    rot_y = wave_x * np.sin(angle) + wave_y * np.cos(angle) + y0
    return rot_x, rot_y

def draw_arrow_line(ax, x0, y0, x1, y1, p, color, label=None, label_pos=None, reverse_arrow=False):
    curr_x, curr_y = x0 + (x1-x0)*p, y0 + (y1-y0)*p
    ax.plot([x0, curr_x], [y0, curr_y], color=color, lw=2.5)
    if p > 0.5:
        mx, my = x0 + (x1-x0)*0.55, y0 + (y1-y0)*0.55
        dx, dy = (x1-x0)*0.01, (y1-y0)*0.01
        astyle = '<-' if reverse_arrow else '->'
        ax.annotate('', xy=(mx+dx, my+dy), xytext=(mx, my),
                    arrowprops=dict(arrowstyle=astyle, color=color, lw=2, mutation_scale=15))
    if p >= 0.95 and label:
        lx, ly = (x1, y1) if label_pos is None else label_pos
        ax.text(lx, ly, label, color='white', fontsize=11)

def update(frame):
    ax.cla()
    ax.set_facecolor(COLORS['bg'])
    ax.set_xlim(-4.5, 4.5)
    ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
    ax.axis('off')

    # Persistent Main Title (Always Visible at the very top)
    ax.text(0, 2.7, "QED: THE LIGHT & MATTER UNIVERSE", color='white', ha='center', fontsize=20, weight='bold')

    scene_idx = frame // (FPS * DURATION_PER_SCENE)
    f = frame % (FPS * DURATION_PER_SCENE)
    alpha_scene = np.clip(f/15 if f < 15 else (270-f)/15 if f > 255 else 1, 0, 1)

    titles = ["Compton Scattering", "Electron-Positron Annihilation", "Pair Production",
              "Bremsstrahlung", "Møller Scattering", "Vacuum Polarization"]
    # Moved Scene Title slightly down to avoid overlapping the main title
    ax.text(-4.3, 2.3, titles[scene_idx], color=COLORS['photon'], fontsize=12, alpha=alpha_scene)

    if scene_idx == 0: # Compton Scattering
        ax.text(0, -2.7, "A photon scatters from an electron, exchanging energy and momentum.", color='white', ha='center', fontsize=11, alpha=alpha_scene)
        off = -0.6
        p1 = np.clip(f/40, 0, 1)
        draw_arrow_line(ax, -3.0+off, 1.5, -1.0+off, 0, p1, COLORS['electron'], r'$e^-$')
        p2 = np.clip((f-10)/40, 0, 1)
        wx, wy = get_sine_wave(-3.0+off, -1.5, -1.0+off, 0, p=p2, reverse=True)
        ax.plot(wx, wy, color=COLORS['photon'], lw=1.5)
        if p2 >= 0.95: ax.text(-3.3+off, -1.7, r'$\gamma$', color='white')
        p3 = np.clip((f-50)/30, 0, 1)
        draw_arrow_line(ax, -1.0+off, 0, 1.0+off, 0, p3, COLORS['electron'])
        p4 = np.clip((f-80)/40, 0, 1)
        draw_arrow_line(ax, 1.0+off, 0, 3.0+off, -1.5, p4, COLORS['electron'], r'$e^-$')
        p5 = np.clip((f-90)/40, 0, 1)
        wx2, wy2 = get_sine_wave(1.0+off, 0, 3.0+off, 1.5, p=p5)
        ax.plot(wx2, wy2, color=COLORS['photon'], lw=1.5)
        if p5 >= 0.95: ax.text(3.1+off, 1.6, r"$\gamma'$", color='white')

    elif scene_idx == 1: # Annihilation
        ax.text(0, -2.7, "An electron and positron annihilate into photons.", color='white', ha='center', fontsize=11, alpha=alpha_scene)
        scale = 0.72
        p1 = np.clip(f/60, 0, 1)
        draw_arrow_line(ax, -3.5*scale, -2.5*scale, 0, 0, p1, COLORS['electron'], r'$e^-$')
        draw_arrow_line(ax, -3.5*scale, 2.5*scale, 0, 0, p1, COLORS['positron'], r'$e^+$', reverse_arrow=True)
        if f >= 60:
            flash_f = f - 60
            if flash_f < 24:
                rad = 0.2 * (flash_f/12 if flash_f < 12 else 1)
                alpha_f = 1 if flash_f < 12 else np.clip(1-(flash_f-12)/12, 0, 1)
                ax.add_patch(plt.Circle((0,0), rad, color='white', alpha=alpha_f))
            if flash_f > 12:
                lbl_f = flash_f - 12
                alpha_lbl = np.clip(lbl_f/10, 0, 1) if lbl_f < 85 else np.clip(1-(lbl_f-85)/20, 0, 1)
                ax.text(0, 0.4, r"$E = mc^2$", color='#ffd54f', ha='center', fontsize=14, alpha=alpha_lbl)
            p2 = np.clip((f-75)/60, 0, 1)
            for ang in [np.pi/4, -np.pi/4]:
                wx, wy = get_sine_wave(0,0, 3.5*scale, 3.5*scale*np.tan(ang), p=p2)
                ax.plot(wx, wy, color=COLORS['photon'], lw=1.5)
                if p2 >= 0.95: ax.text(3.6*scale, 3.6*scale*np.tan(ang), r'$\gamma$', color='white')

    elif scene_idx == 2: # Pair Production
        ax.text(0, -2.7, "A high-energy photon creates a matter-antimatter pair near a nucleus", color='white', ha='center', fontsize=11, alpha=alpha_scene)
        ax.add_patch(plt.Circle((-2, 0), 0.9, color=COLORS['nucleus'], alpha=0.3))
        for ang in np.linspace(-0.5, 0.5, 5):
            t = np.linspace(-4, 4, 100); ax.plot(t, 0.2*np.sin(t+ang), color='white', alpha=0.04, ls='--')
        p1 = np.clip(f/60, 0, 1)
        wx, wy = get_sine_wave(-4.5, 0, -2.9, 0, p=p1)
        ax.plot(wx, wy, color=COLORS['photon'], lw=1.5)
        if f >= 60:
            if f < 75: ax.add_patch(plt.Circle((-2.9, 0), 0.15, color='white', alpha=1-(f-60)/15))
            p2 = np.clip((f-60)/80, 0, 1)
            theta = np.linspace(0, 0.8*p2, 50)
            ax.plot(-2.9 + 3.5*np.sin(theta), 3.5-3.5*np.cos(theta), color=COLORS['electron'], lw=2.5)
            ax.plot(-2.9 + 3.5*np.sin(theta), -3.5+3.5*np.cos(theta), color=COLORS['positron'], lw=2.5)

    elif scene_idx == 3: # Bremsstrahlung
        ax.text(0, -2.7, "An electron braking near a nucleus radiates a photon", color='white', ha='center', fontsize=11, alpha=alpha_scene)
        ax.add_patch(plt.Circle((0, -1.5), 0.7, color=COLORS['nucleus'], alpha=0.4))
        p1 = np.clip(f/100, 0, 1)
        t = np.linspace(0, np.pi * p1, 200)
        x, y = -3.5 + 7*t/np.pi, -0.8*np.sin(t)
        ax.plot(x, y, color=COLORS['electron'], lw=2.5)

    elif scene_idx == 4: # Møller Scattering
        ax.text(0, -2.7, "Two electrons repel each other by exchanging a virtual photon", color='white', ha='center', fontsize=11, alpha=alpha_scene)
        vx_x = 0; v_up, v_dn = 1.0, -1.0
        p1 = np.clip(f/60, 0, 1)
        draw_arrow_line(ax, -3.5, 1.8, vx_x, v_up, p1, COLORS['electron'], r'$e^-$')
        draw_arrow_line(ax, -3.5, -1.8, vx_x, v_dn, p1, COLORS['electron'], r'$e^-$')
        if f > 60:
            ax.add_patch(plt.Circle((vx_x, v_up), 0.08, color='white'))
            ax.add_patch(plt.Circle((vx_x, v_dn), 0.08, color='white'))
            alpha_v = 0.4 + 0.5 * np.abs(np.sin(f*0.2))
            ax.plot([vx_x, vx_x], [v_up, v_dn], color=COLORS['photon'], ls='--', lw=2, alpha=alpha_v)
            ax.text(vx_x+0.2, 0, r"virtual $\gamma$", color=COLORS['photon'], fontsize=9, alpha=alpha_v)
            p2 = np.clip((f-80)/80, 0, 1)
            draw_arrow_line(ax, vx_x, v_up, 3.5, 1.8, p2, COLORS['electron'], r'$e^-$')
            draw_arrow_line(ax, vx_x, v_dn, 3.5, -1.8, p2, COLORS['electron'], r'$e^-$')

    elif scene_idx == 5: # Vacuum Polarization
        ax.text(0, -2.7, "The quantum vacuum itself modifies how photons travel", color='white', ha='center', fontsize=11, alpha=alpha_scene)
        wx, wy = get_sine_wave(-4.5, 0, 4.5, 0)
        ax.plot(wx, wy, color='#ffd54f', lw=1.5, alpha=0.6)
        p1 = np.clip(f/80, 0, 1)
        theta = np.linspace(0, 2*np.pi*p1, 200)
        r = 1.1
        ax.plot(r*np.cos(theta[theta<=np.pi]), r*np.sin(theta[theta<=np.pi]), color=COLORS['electron'], lw=3)
        ax.plot(r*np.cos(theta[theta>np.pi]), r*np.sin(theta[theta>np.pi]), color=COLORS['positron'], lw=3)

    return []

ani = FuncAnimation(fig, update, frames=TOTAL_FRAMES, interval=1000/FPS)
writer = FFMpegWriter(fps=FPS, bitrate=2500)
ani.save('qed_light_matter.mp4', writer=writer)
plt.close()
mp4 = open('qed_light_matter.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
display(HTML(f'<video width=960 controls autoplay loop><source src="{data_url}" type="video/mp4"></video>'))
files.download('qed_light_matter.mp4')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>